# 30. Franchise mode: the engine (no UI yet)

**Franchise mode** turns the simulator into a loop you can play: pick a team, set its lineup, play a stretch of games
(the next N games, a date range, or the rest of the season), and look at the results two ways:

- **One season, as a story:** the team's game log, record, standings, scoring leaders and goalie lines. It's one
  possible version of those games, seeded: the same choices and seed always give the same story, and a new seed
  re-rolls it.
- **10,000 seasons, as a projection:** the same choices played 10,000 times, so you see the range (10th to 90th
  percentile) and, for a full season, the playoff odds.

Everything runs on the simulator that notebooks 24-29 built and checked. `src/franchise.py` only decides what goes in
(the lineups and the window of games) and how the results are shown. Player progression (an OVR change at the end of
a season) is notebook 31, so the story keeps every player's random season for it.

This notebook writes nothing.

## What a lineup change can and can't do

In this model a team's strength is **the sum of each dressed player's projected value (WAR per 60 minutes) x his
minutes**. So a lineup change matters only through two things:

1. **Who dresses.** Putting a better player in instead of a weaker one helps by the gap in their value x the minutes
   of that slot.
2. **How many minutes each player gets.** Minutes follow the slot: F1 plays the most among forwards, F12 the least.
   Moving a good player up gives him more minutes, so the team gets a little stronger.

What it **can't** do: **who a player's linemates are doesn't matter.** There is no chemistry in the model, and this
notebook doesn't invent any. A "line" here is just three slots in a row (F1-F3 is line 1, the three forwards with the
most minutes), and a D pair is two slots (D1-D2). The C/W line labels from notebook 23 were only labels: minutes
always followed rank.

The starting goalie (G1) gets the starter's games; the backup (G2) starts the second night of back-to-backs plus
enough games to make about 30 of 84. You can also set a team's start share.

In [1]:
import sys
sys.path.append("../src")

import subprocess
import time
import types
from pathlib import Path
import numpy as np
import pandas as pd

import lineups as L
import season as SE
import refresh as R
import franchise as FR

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
BUNDLE_DIR = DATA / "simulator_2026-27"
TRADES = DATA / "trades_2026-27.csv"
CACHE_ROOT = BUNDLE_DIR / "updated"
SIT = list(pd.read_csv(DATA / "sit_out_2026-27.csv")["playerId"])     # Larkin and Edvinsson sit, as in notebook 29
TEAM = "DET"

## 1. Inputs: today, with the 2026-27 results so far

`FR.load_inputs` follows the simulator switch's rules. **With the update** (notebook 29's newest cache): real
results through its evidence date, each player's updated draws, and the updated team bonus; it's only allowed from
the cache's start date on (from an earlier date the update would already know games being simulated). **Without
it:** preseason ratings.

In [2]:
cache_dir = R.newest_cache(CACHE_ROOT)
meta = R.load_update_cache(cache_dir, SE.load_bundle(BUNDLE_DIR, TRADES), BUNDLE_DIR)[2]
START = meta["start"]
inp = FR.load_inputs(BUNDLE_DIR, TRADES, START, use_update=True, cache_root=CACHE_ROOT)
B = inp["bundle"]
print(f"Start {START}. Ratings: {inp['ratings']}. Trades in effect: {', '.join(SE.default_trades(B['trades'], START))}.")
print("Sitting out:", ", ".join(B["players"].set_index("playerId").loc[SIT, "name"]))
try:
    FR.load_inputs(BUNDLE_DIR, TRADES, "2026-11-01", use_update=True, cache_root=CACHE_ROOT)
except ValueError as e:
    print("Refused, as it should be:", e)

Start 2026-11-16. Ratings: updated through 2026-11-15 (notebook 29). Trades in effect: T1, T2, T3, T4, T5, T6, W1.
Sitting out: Dylan Larkin, Simon Edvinsson
Refused, as it should be: the update through 2026-11-15 can only be used from 2026-11-16 on: from an earlier start it would already know games being simulated


## 2. Detroit's lineup as a slot table

A franchise lineup is a small table: **team, slot, playerId**, one row per slot (F1-F12, D1-D6, G1, G2). That's the
format the app can write later (`FR.write_overrides` / `FR.read_overrides`; a G1 row can carry a `start_share`).
`FR.lineup_table` gives today's 20 slots. Passed back unchanged, it gives the baseline exactly (checked in section 5).

In [3]:
base = FR.base_lineups(inp, sit_out=SIT)
det = FR.lineup_table(base, TEAM)
show = det.merge(base[["playerId", "status", "minutes_per_game", "proj_war_60"]], on="playerId")
show.round({"minutes_per_game": 1, "proj_war_60": 3})

,team,slot,playerId,name,position,status,minutes_per_game,proj_war_60
0,DET,F1,8482078,Lucas Raymond,L,NHL roster,19.4,0.114
1,DET,F2,8479337,Alex DeBrincat,R,NHL roster,18.9,0.119
2,DET,F3,8484471,Emmitt Finnie,C,NHL roster,16.4,0.043
3,DET,F4,8477429,Andrew Copp,C,NHL roster,17.0,0.004
4,DET,F5,8478042,Viktor Arvidsson,L,NHL roster,15.3,0.059
5,DET,F6,8477456,J.T. Compher,L,NHL roster,16.0,0.015
6,DET,F7,8483464,Marco Kasper,C,NHL roster,14.4,0.029
7,DET,F8,8478891,Mason Appleton,C,NHL roster,14.0,0.009
8,DET,F9,8479992,Michael Rasmussen,C,NHL roster,13.1,0.034
9,DET,F10,8484794,Michael Brandsegg-Nygård,R,NHL roster,12.8,0.028


In [4]:
pool = base[(base["team"] == TEAM) & ~base["dressed"]]
print("Not dressed today (the call-up candidates are the AHL/ECHL ones):")
pool[["name", "position", "status", "proj_war_60"]].sort_values("proj_war_60", ascending=False).round(3).head(10)

Not dressed today (the call-up candidates are the AHL/ECHL ones):


,name,position,status,proj_war_60
408,Michal Postava,G,NHL roster,0.044
383,Carter Bear,L,AHL/ECHL (NHL contract),0.043
409,Trey Augustine,G,AHL/ECHL (NHL contract),0.042
384,Jesse Kiiskinen,R,AHL/ECHL (NHL contract),0.036
401,Axel Sandin-Pellikka,D,AHL/ECHL (NHL contract),0.032
380,Nate Danielson,C,NHL roster,0.031
405,Jacob Bryson,D,AHL/ECHL (NHL contract),0.026
402,Shai Buium,D,AHL/ECHL (NHL contract),0.023
385,Sutter Muzzatti,C,AHL/ECHL (NHL contract),0.019
386,Ondrej Becher,C,AHL/ECHL (NHL contract),0.018


## 3. Minutes when you move players: three rules

**Your rule ("moved", the default):** a player whose slot you changed gets **100% slot minutes**; everyone you didn't
touch keeps the usual **75% own recent ice time + 25% slot** blend (notebook 24); then the team's forwards and
defensemen are rescaled so their totals stay the same. For comparison: **"blend"** (everyone keeps the blend) and
**"slot"** (the whole team gets slot minutes only).

The example: Detroit swaps its 1st and 4th lines (F1-F3 trade places with F10-F12).

In [5]:
swap = FR.swap_lines(det, TEAM, "F", 1, 4)
dressed_f = base[(base["team"] == TEAM) & base["dressed"] & (base["group"] == "F")].sort_values("rank")
cmp = dressed_f[["name", "slot", "minutes_per_game"]].rename(columns={"slot": "old slot", "minutes_per_game": "baseline"})
war = {"baseline": base.loc[(base["team"] == TEAM) & base["dressed"], "war_per_game"].sum()}
totals = {}
for rule in FR.MINUTES_RULES:
    lu, _, _ = FR.apply_overrides(base, B, swap, minutes_rule=rule)
    d = lu[(lu["team"] == TEAM) & lu["dressed"]]
    cmp[rule] = cmp.index.map(lu["minutes_per_game"])
    if rule == "moved":
        cmp.insert(2, "new slot", cmp.index.map(lu["slot"]))
    war[rule] = d["war_per_game"].sum()
    totals[rule] = (d.loc[d["group"] == "F", "minutes_per_game"].sum(), d.loc[d["group"] == "D", "minutes_per_game"].sum())
print("Minutes per game, Detroit forwards, lines 1 and 4 swapped:")
display(cmp.round(1))
print("Team WAR per game:", {k: round(float(v), 4) for k, v in war.items()})
print("Change over 84 games (WAR):", {k: round(float(v - war["baseline"]) * 84, 2) for k, v in war.items() if k != "baseline"})
print("Forward and defense minutes per game after the swap (they don't change):",
      {k: (round(float(f), 2), round(float(d), 2)) for k, (f, d) in totals.items()})

Minutes per game, Detroit forwards, lines 1 and 4 swapped:


,name,old slot,new slot,baseline,moved,blend,slot
368,Lucas Raymond,F1,F10,19.4,12.5,17.2,12.4
369,Alex DeBrincat,F2,F11,18.9,12.0,16.8,11.9
370,Emmitt Finnie,F3,F12,16.4,11.4,14.4,11.2
371,Andrew Copp,F4,F4,17.0,16.8,16.6,16.7
372,Viktor Arvidsson,F5,F5,15.3,15.2,14.9,16.0
373,J.T. Compher,F6,F6,16.0,15.8,15.6,15.4
374,Marco Kasper,F7,F7,14.4,14.2,14.0,14.7
375,Mason Appleton,F8,F8,14.0,13.8,13.6,13.9
376,Michael Rasmussen,F9,F9,13.1,13.0,12.8,13.2
377,Michael Brandsegg-Nygård,F10,F1,12.8,19.5,14.2,19.3


Team WAR per game: {'baseline': 0.2194, 'moved': 0.1972, 'blend': 0.2117, 'slot': 0.1969}
Change over 84 games (WAR): {'moved': -1.86, 'blend': -0.64, 'slot': -1.89}
Forward and defense minutes per game after the swap (they don't change): {'moved': (180.62, 116.62), 'blend': (180.62, 116.62), 'slot': (180.62, 116.62)}


## 4. Validation: what blocks a lineup, what only warns

**Blocks:** a team that doesn't list F1-F12, D1-D6, G1 and G2 exactly once; a player listed twice; a player who isn't
in that team's pool (another team's player, or one sitting out); a goalie in a skater slot or a skater in a goalie
slot; a start share outside 0 to 1. **Warns:** fewer than 4 natural centers, a skater out of position, a call-up dressed.

In [6]:
pid = lambda name: int(B["players"].loc[B["players"]["name"] == name, "playerId"].iloc[0])
bad = det.copy()
bad.loc[bad["slot"] == "F12", "slot"] = "F13"                                    # a slot that doesn't exist
bad = FR.replace(bad, TEAM, pid("Andrew Copp"), pid("Auston Matthews"))           # another team's player
bad = FR.replace(bad, TEAM, pid("Mason Appleton"), pid("Michal Postava"))         # a goalie at forward
bad = FR.replace(bad, TEAM, pid("Lucas Raymond"), pid("Dylan Larkin"))            # sitting out
try:
    FR.apply_overrides(base, B, bad)
except ValueError as e:
    print(e)
print()
odd = FR.replace(det, TEAM, pid("Andrew Copp"), pid("Jacob Bernard-Docker"))      # a defenseman at forward
odd = FR.replace(odd, TEAM, pid("Marco Kasper"), pid("Carter Bear"))              # a call-up, one center fewer
print("Warnings (the run goes ahead):")
for w in FR.apply_overrides(base, B, odd)[2]:
    print(" -", w)
print("Start share 1.3:", FR.check_overrides(base, None, {TEAM: 1.3})[0])

The franchise lineup can't be used:
  - DET: needs F1-F12, D1-D6, G1 and G2 once each; missing ['F12']; extra or twice ['F13']
  - DET F1: Dylan Larkin isn't in DET's pool (he's sitting out)
  - DET F4: Auston Matthews isn't in DET's pool (he's TOR's)
  - DET F8: Michal Postava is a goalie; goalies and skaters can't trade slots

Warnings (the run goes ahead):
 - DET F4: Jacob Bernard-Docker (D) plays out of position; he gets F minutes and keeps his own value
 - DET: only 2 natural centers among the forwards (the coach rule wants 4); the model doesn't penalize it, but it isn't a real lineup
 - DET: call-ups dressed: Carter Bear
Start share 1.3: ['DET: start share 1.3 must be between 0 and 1']


## 5. Check: no overrides reproduces run_scenario exactly

Three checks:

1. **Franchise mode with no overrides over the full season = `season.run_scenario`**, bit for bit (team table,
   tallies, seeds, every season's points, skater and goalie lines, lineups), with and without the update cache, for
   one season and for 10,000.
2. **The new options don't change anything by default:** the edited `season.py` and `lineups.py` give the same
   results, bit for bit, as the committed versions (read from git, run in memory).
3. **A baseline slot table passed as an override** gives the baseline minutes exactly.

One note: a one-season *story* moves updated players to a new draw when the seed changes (`update_shift`; before,
season 1 always used each updated player's first draw, whatever the seed). At seed 0 that's the same as before, so
the story check uses seed 0.

In [7]:
def same(x, y):
    if isinstance(x, dict):
        return set(x) == set(y) and all(same(x[k], y[k]) for k in x)
    if isinstance(x, pd.DataFrame):
        return x.equals(y)
    return np.array_equal(x, y, equal_nan=True)

KEYS = ["table", "tallies", "seeds", "points", "skaters", "goalies", "lineups"]
inp_pre = FR.load_inputs(BUNDLE_DIR, TRADES, "2026-11-05", use_update=False, cache_root=CACHE_ROOT)
rows = []
for label, I in [("preseason, from Nov 5", inp_pre), ("updated, from " + START, inp)]:
    for n, seed in [(1, 0), (10000, 3)]:
        ref = SE.run_scenario(I["bundle"], start_date=I["start"], sit_out=SIT, n_sims=n, seed=seed, stat_lines=True,
                              updated=I["updated"])
        mine = FR.play(I, TEAM, sit_out=SIT, n_sims=n, seed=seed, story=(n == 1))
        rows.append({"inputs": label, "seasons": n, "seed": seed, "identical": all(same(ref[k], mine[k]) for k in KEYS)})
check1 = pd.DataFrame(rows)
check1

,inputs,seasons,seed,identical
0,"preseason, from Nov 5",1,0,True
1,"preseason, from Nov 5",10000,3,True
2,"updated, from 2026-11-16",1,0,True
3,"updated, from 2026-11-16",10000,3,True


In [8]:
def head_module(name, deps):
    """The committed (HEAD) version of src/<name>.py, loaded in memory (no files written)."""
    src = subprocess.run(["git", "show", f"HEAD:src/{name}.py"], capture_output=True, text=True, check=True).stdout
    mod = types.ModuleType(f"{name}_head")
    saved = {k: sys.modules.get(k) for k in deps}
    sys.modules.update(deps)
    try:
        exec(compile(src, f"HEAD:src/{name}.py", "exec"), mod.__dict__)
    finally:
        for k, v in saved.items():
            sys.modules[k] = v
    return mod

L_head = head_module("lineups", {})
SE_head = head_module("season", {"lineups": L_head})
rows = []
for label, I in [("preseason, from Nov 5", inp_pre), ("updated, from " + START, inp)]:
    for n in [1, 10000]:
        a = SE_head.run_scenario(I["bundle"], start_date=I["start"], sit_out=SIT, n_sims=n, seed=7, stat_lines=True, updated=I["updated"])
        b = SE.run_scenario(I["bundle"], start_date=I["start"], sit_out=SIT, n_sims=n, seed=7, stat_lines=True, updated=I["updated"])
        rows.append({"inputs": label, "seasons": n, "committed = edited": all(same(a[k], b[k]) for k in KEYS)})
check2 = pd.DataFrame(rows)
display(check2)
as_override = pd.concat([FR.lineup_table(base, t) for t in base["team"].unique()])
lu_same, _, _ = FR.apply_overrides(base, B, as_override)
check3 = np.array_equal(lu_same["minutes_per_game"].to_numpy(), base["minutes_per_game"].to_numpy())
print("All 32 baseline slot tables passed as overrides give the baseline minutes exactly:", check3)
assert check1["identical"].all() and check2["committed = edited"].all() and check3

,inputs,seasons,committed = edited
0,"preseason, from Nov 5",1,True
1,"preseason, from Nov 5",10000,True
2,"updated, from 2026-11-16",1,True
3,"updated, from 2026-11-16",10000,True


All 32 baseline slot tables passed as overrides give the baseline minutes exactly: True


## 6. One season: Detroit's next 10 games

`FR.story` plays the window once. Every team plays its own games in the window (Detroit's 10 games end on a date;
every game league-wide through that date is played). Player stats are shown **two ways**: **range only** (just these
games) and **season to date** (real stats before the start + these games, the same rule as the simulator page).

**What's simulated:** goals, assists, wins and goals against. **Not simulated:** shots, save %, ice time, +/-, so they
aren't shown. GF counts goals players scored: a shootout win is +1 in the score but not in GF.

In [9]:
s10 = FR.story(inp, TEAM, n_games=10, sit_out=SIT, seed=0)
r = s10["record"]
print(f"{TEAM}, {s10['range']['start']} to {s10['range']['end']} ({s10['range']['team_games']} games; "
      f"{s10['range']['games']} games league-wide). Seed {s10['seed']}. Ratings: {s10['ratings']}.")
print(f"Record {r['wins']}-{r['losses']}-{r['otl']}, {r['points']} points, GF {r['gf']}, GA {r['ga']}, "
      f"pace {r['pace_84']:.0f} points per 84 games.")
s10["game_log"]

DET, 2026-11-16 to 2026-12-11 (10 games; 183 games league-wide). Seed 0. Ratings: updated through 2026-11-15 (notebook 29).
Record 3-4-3, 9 points, GF 23, GA 25, pace 76 points per 84 games.


,date,opponent,home_away,score,result,goalie,record,points,gf,ga
0,2026-11-18,BOS,home,0-5,L,John Gibson,0-1-0,0,0,5
1,2026-11-21,NYI,home,3-4 OT,OTL,John Gibson,0-1-1,1,3,4
2,2026-11-25,VAN,home,1-2,L,John Gibson,0-2-1,1,1,2
3,2026-11-27,CBJ,away,1-2 OT,OTL,Daniil Tarasov,0-2-2,2,1,2
4,2026-11-28,NSH,home,2-3,L,Daniil Tarasov,0-3-2,2,2,3
5,2026-12-01,CGY,home,8-2,W,John Gibson,1-3-2,4,8,2
6,2026-12-03,COL,home,1-2 OT,OTL,John Gibson,1-3-3,5,1,2
7,2026-12-05,BOS,away,0-4,L,John Gibson,1-4-3,5,0,4
8,2026-12-09,NJD,away,3-1,W,Daniil Tarasov,2-4-3,7,3,1
9,2026-12-11,NYR,home,4-0,W,John Gibson,3-4-3,9,4,0


In [10]:
st10 = s10["standings"]
print("Standings at the end of the range (real games before the start + these games; no seeds or odds, since the")
print("season isn't over). Sorted by points, then regulation wins, then regulation + overtime wins:")
st10[["team", "division", "gp", "wins", "losses", "otl", "points", "goal_diff"]].head(12)

Standings at the end of the range (real games before the start + these games; no seeds or odds, since the
season isn't over). Sorted by points, then regulation wins, then regulation + overtime wins:


,team,division,gp,wins,losses,otl,points,goal_diff
0,OTT,Atlantic,34,25,6,3,53,29
1,EDM,Pacific,34,23,7,4,50,31
2,CAR,Metropolitan,31,22,6,3,47,53
3,MIN,Central,34,20,8,6,46,27
4,NSH,Central,32,20,7,5,45,16
5,MTL,Atlantic,33,18,7,8,44,11
6,TBL,Atlantic,30,20,7,3,43,26
7,NJD,Metropolitan,32,21,10,1,43,26
8,VGK,Pacific,33,19,10,4,42,11
9,COL,Central,31,18,8,5,41,29


In [11]:
sk_cols = {"name": "Player", "position": "Pos", "sim_gp": "GP", "sim_g": "G", "sim_a": "A", "sim_p": "P"}
print("Detroit skaters, RANGE ONLY (these 10 games):")
display(s10["team_skaters"]["range"][list(sk_cols)].rename(columns=sk_cols).head(10))
season_cols = {"name": "Player", "position": "Pos", "real_gp": "Real GP", "real_p": "Real P", "gp": "GP", "g": "G", "a": "A", "p": "P"}
print(f"Detroit skaters, SEASON TO DATE ({s10['season_label']}):")
display(s10["team_skaters"]["season"][list(season_cols)].rename(columns=season_cols).head(10))
g_range = {"name": "Goalie", "role": "Role", "sim_gp": "GP", "sim_wins": "W", "sim_losses": "L", "sim_otl": "OTL", "sim_ga": "GA", "sim_gaa": "GAA"}
g_season = {"name": "Goalie", "role": "Role", "gp": "GP", "w": "W", "l": "L", "otl": "OTL", "ga": "GA", "gaa": "GAA"}
print("Detroit goalies, range only, then season to date (GAA leaves out empty-net and shootout goals; a start = 60 minutes):")
display(s10["team_goalies"]["range"][list(g_range)].rename(columns=g_range).round(2))
display(s10["team_goalies"]["season"][list(g_season)].rename(columns=g_season).round(2))

Detroit skaters, RANGE ONLY (these 10 games):


,Player,Pos,GP,G,A,P
163,Alex DeBrincat,R,10,5,3,8
162,Lucas Raymond,L,10,2,6,8
173,Carter Mazur,L,10,2,6,8
169,Mason Appleton,C,10,3,3,6
165,Andrew Copp,C,10,1,4,5
174,Moritz Seider,D,10,1,4,5
166,Viktor Arvidsson,L,10,1,3,4
177,Anton Johansson,D,10,0,4,4
167,J.T. Compher,L,10,3,0,3
164,Emmitt Finnie,C,10,1,2,3


Detroit skaters, SEASON TO DATE (season to date (real before 2026-11-16 + the range)):


,Player,Pos,Real GP,Real P,GP,G,A,P
45,Lucas Raymond,L,21,21,31,10,19,29
78,Alex DeBrincat,R,21,16,31,12,12,24
141,Viktor Arvidsson,L,21,15,31,9,10,19
155,Moritz Seider,D,21,13,31,2,16,18
208,Andrew Copp,C,21,10,31,4,11,15
225,Emmitt Finnie,C,21,11,31,8,6,14
259,Carter Mazur,L,14,5,24,3,10,13
269,Michael Brandsegg-Nygård,R,21,11,31,4,8,12
339,J.T. Compher,L,14,7,24,6,4,10
334,Justin Faulk,D,21,9,31,3,7,10


Detroit goalies, range only, then season to date (GAA leaves out empty-net and shootout goals; a start = 60 minutes):


,Goalie,Role,GP,W,L,OTL,GA,GAA
18,John Gibson,Starter,7,2,3,2,19,2.71
19,Daniil Tarasov,Backup,3,1,1,1,6,2.00


,Goalie,Role,GP,W,L,OTL,GA,GAA
14,John Gibson,Starter,20,6,10,3,60,3.25
46,Daniil Tarasov,Backup,12,4,4,3,21,1.88
79,Michal Postava,Not dressed,2,1,0,0,4,2.89


In [12]:
lead = lambda d, cols: d[["name", "team"] + cols].reset_index(drop=True)
print("League scoring leaders, range only (points):")
display(lead(s10["leaders"]["range"]["points"], ["sim_g", "sim_a", "sim_p"]).head(5))
print("League scoring leaders, season to date (points):")
display(lead(s10["leaders"]["season"]["points"], ["g", "a", "p"]).head(5))

League scoring leaders, range only (points):


,name,team,sim_g,sim_a,sim_p
0,Connor McDavid,EDM,9,19,28
1,Leon Draisaitl,EDM,6,16,22
2,Kyle Connor,WPG,9,9,18
3,Quinn Hughes,MIN,2,14,16
4,Clayton Keller,UTA,7,7,14


League scoring leaders, season to date (points):


,name,team,g,a,p
0,Connor McDavid,EDM,21,60,81
1,Leon Draisaitl,EDM,26,37,63
2,Quinn Hughes,MIN,9,37,46
3,Kyle Connor,WPG,26,18,44
4,Kirill Kaprizov,MIN,20,24,44


### Checks: does the story add up?

For every team: **team GF = the sum of its players' goals**, **points = 2 x wins + OT losses**, and **games played in
the range = its scheduled games in the window**. Detroit plays exactly 10. And for **every player**: **season to date =
real stats so far + the range** (skaters GP, G, A, P; goalies GP, W, L, OTL, GA), with real stats read on their own
from the boxscores (`season.real_lines`).

In [13]:
def story_checks(s, n_expected=None):
    tr = s["teams_range"].set_index("team")
    goals = s["range_skaters"].groupby("team")["sim_g"].sum().reindex(tr.index).fillna(0)
    sched = s["run"]["bundle_used"]["schedule"]
    w = sched[(sched["date"] >= s["range"]["start"]) & (sched["date"] <= s["range"]["end"])]
    sched_gp = pd.concat([w["home"], w["away"]]).value_counts().reindex(tr.index).fillna(0)
    real_s, real_g = SE.real_lines(s["run"]["bundle_used"], s["range"]["start"])
    ss, rs = s["season_skaters"].set_index("playerId"), s["range_skaters"].set_index("playerId")
    rr = real_s.set_index("playerId")
    sk_ok = all(np.allclose(ss[c], rs[f"sim_{c}"].reindex(ss.index).fillna(0) + rr[f"real_{c}"].reindex(ss.index).fillna(0),
                            rtol=0, atol=1e-9) for c in ["gp", "g", "a", "p"])
    gs, gr, gq = s["season_goalies"].set_index("playerId"), s["range_goalies"].set_index("playerId"), real_g.set_index("playerId")
    pairs = [("gp", "sim_gp", "real_gp"), ("w", "sim_wins", "real_w"), ("l", "sim_losses", "real_l"),
             ("otl", "sim_otl", "real_otl"), ("ga", "sim_ga", "real_ga")]
    g_ok = all(np.allclose(gs[a], gr[b].reindex(gs.index).fillna(0) + gq[c].reindex(gs.index).fillna(0), rtol=0, atol=1e-9)
               for a, b, c in pairs)
    out = {"GF = player goals (all 32)": bool((tr["gf"] == goals).all()),
           "points = 2W + OTL (all 32)": bool((tr["points"] == 2 * tr["wins"] + tr["otl"]).all()
                                              and (s["standings"]["points"] == 2 * s["standings"]["wins"] + s["standings"]["otl"]).all()),
           "GP = scheduled games in the window (all 32)": bool((tr["gp"] == sched_gp).all()),
           f"season = real + range ({len(ss)} skaters)": sk_ok, f"season = real + range ({len(gs)} goalies)": g_ok}
    if n_expected is not None:
        out[f"{s['team']} plays exactly {n_expected}"] = len(s["game_log"]) == n_expected
    return out

c10 = story_checks(s10, 10)
for k, v in c10.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(c10.values())

OK  GF = player goals (all 32)
OK  points = 2W + OTL (all 32)
OK  GP = scheduled games in the window (all 32)
OK  season = real + range (791 skaters)
OK  season = real + range (80 goalies)
OK  DET plays exactly 10


### Re-roll: a new seed is a new story

Same choices, seed 1. Player luck re-rolls too: the updated players' draws move with the seed.

In [14]:
s10b = FR.story(inp, TEAM, n_games=10, sit_out=SIT, seed=1)
for s in [s10, s10b]:
    r = s["record"]
    top = s["team_skaters"]["range"].iloc[0]
    print(f"seed {s['seed']}: {r['wins']}-{r['losses']}-{r['otl']}, {r['points']} points, GF {r['gf']}, GA {r['ga']}; "
          f"top scorer {top['name']} {top['sim_g']}G {top['sim_a']}A")
dz = s10["draws"].set_index("playerId")["z"] - s10b["draws"].set_index("playerId")["z"]
print(f"Players whose random season changed between the two seeds: {(dz != 0).sum()} of {len(dz)}")

seed 0: 3-4-3, 9 points, GF 23, GA 25; top scorer Alex DeBrincat 5G 3A
seed 1: 1-8-1, 3 points, GF 25, GA 44; top scorer Lucas Raymond 1G 8A
Players whose random season changed between the two seeds: 1342 of 1348


## 7. One season: the rest of the season (final season totals)

When the window runs to the last game, the story is a **whole season**: final standings with playoff seeds, and the
season-to-date lines are the **final season totals**.

In [15]:
sF = FR.story(inp, TEAM, sit_out=SIT, seed=0)
r = sF["record"]
me = sF["standings"].set_index("team").loc[TEAM]
print(f"{TEAM} from {START}: {r['wins']}-{r['losses']}-{r['otl']} in {r['gp']} games, {r['points']} points (GF {r['gf']}, GA {r['ga']}).")
print(f"Final: {me['wins']}-{me['losses']}-{me['otl']}, {me['points']} points, league rank {me['league_rank']}, "
      f"seed {me['seed'] or 'out of the playoffs'}. Lines below are {sF['season_label']}.")
display(sF["standings"][sF["standings"]["division"] == "Atlantic"][["seed", "team", "gp", "wins", "losses", "otl", "points", "goal_diff"]])
display(sF["team_skaters"]["season"][list(season_cols)].rename(columns=season_cols).head(8))
display(sF["team_goalies"]["season"][list(g_season)].rename(columns=g_season).round(2))
print("League leaders, final season totals:")
display(lead(sF["leaders"]["season"]["points"], ["g", "a", "p"]).head(5))
cF = story_checks(sF)
for k, v in cF.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(cF.values())

DET from 2026-11-16: 29-25-9 in 63 games, 67 points (GF 198, GA 211).
Final: 37-35-12, 86 points, league rank 23, seed out of the playoffs. Lines below are final season totals.


,seed,team,gp,wins,losses,otl,points,goal_diff
1,1,MTL,84,56,18,10,122,81
3,2,BOS,84,56,25,3,115,71
4,3,TBL,84,54,24,6,114,83
11,WC1,OTT,84,45,32,7,97,2
14,WC2,FLA,84,42,31,11,95,-6
17,,BUF,84,41,32,11,93,-11
22,,DET,84,37,35,12,86,-23
25,,TOR,84,35,41,8,78,-36


,Player,Pos,Real GP,Real P,GP,G,A,P
48,Lucas Raymond,L,21,21,84,28,43,71
53,Viktor Arvidsson,L,21,15,84,31,38,69
56,Alex DeBrincat,R,21,16,84,31,37,68
129,Moritz Seider,D,21,13,84,4,46,50
141,Emmitt Finnie,C,21,11,84,26,23,49
148,Carter Mazur,L,14,5,77,21,27,48
203,Andrew Copp,C,21,10,84,14,26,40
238,J.T. Compher,L,14,7,77,17,20,37


,Goalie,Role,GP,W,L,OTL,GA,GAA
14,John Gibson,Starter,53,22,23,7,163,3.17
46,Daniil Tarasov,Backup,32,14,12,5,91,2.92
79,Michal Postava,Not dressed,2,1,0,0,4,2.89


League leaders, final season totals:


,name,team,g,a,p
0,Connor McDavid,EDM,43,117,160
1,Nikita Kucherov,TBL,43,92,135
2,Leon Draisaitl,EDM,63,69,132
3,Nathan MacKinnon,COL,37,80,117
4,Quinn Hughes,MIN,20,90,110


OK  GF = player goals (all 32)
OK  points = 2W + OTL (all 32)
OK  GP = scheduled games in the window (all 32)
OK  season = real + range (791 skaters)
OK  season = real + range (80 goalies)


## 8. Every player's random season, kept for notebook 31

Each simulated season gives every pool player one random draw **z** (from his notebook 17 pool, or from his updated
draws when the update is on). In that season his value is **projected WAR/60 + z x his draw size**, and his scoring
rates are multiplied by **exp(k z)** (notebook 27). The story keeps them all, so notebook 31 can turn a simulated
season into an OVR change. Two notes for notebook 31: z is a **whole-season** outcome even when the window is 10
games, and a player who didn't dress has a z but no WAR change (he didn't play).

In [16]:
dr = sF["draws"]
print(f"{len(dr):,} pool players; {dr['dressed'].sum()} dressed; draws from: {dr['draw_from'].value_counts().to_dict()}")
dr[dr["team"] == TEAM].sort_values("war60_change", ascending=False).round(4).head(8)

1,348 pool players; 640 dressed; draws from: {'updated': 1348}


,playerId,name,team,group,dressed,games_in_range,minutes_per_game,z,draw_from,draw_size,war60_change,proj_war_60,season_war60,scoring_multiplier
851,8482802,Carter Mazur,DET,F,True,63,11.5140,1.0347,updated,0.0529,0.0547,0.0186,0.0733,1.1849
369,8478891,Mason Appleton,DET,F,True,63,13.9579,0.9412,updated,0.0498,0.0469,0.0088,0.0557,1.1661
283,8478042,Viktor Arvidsson,DET,F,True,63,15.3225,0.9644,updated,0.0485,0.0468,0.0586,0.1054,1.1732
1186,8484471,Emmitt Finnie,DET,F,True,63,16.3956,0.8084,updated,0.0476,0.0385,0.0425,0.0810,1.1375
1011,8483679,Anton Johansson,DET,D,True,63,18.3007,1.0444,updated,0.0283,0.0295,0.0229,0.0524,1.1876
210,8477456,J.T. Compher,DET,F,True,63,15.9819,0.6022,updated,0.0479,0.0289,0.0155,0.0443,1.0971
108,8476434,John Gibson,DET,G,True,63,38.5961,0.5495,updated,0.0424,0.0233,0.0541,0.0774,1.0868
651,8481607,Albert Johansson,DET,D,True,63,15.5433,0.3725,updated,0.0297,0.0111,0.0087,0.0198,1.0534


## 9. The projection: the same choices, 10,000 seasons

Detroit's next 10 games, then the rest of the season. Ranges are the 10th to 90th percentile. Player lines again two
ways; in the season-to-date view the 10th and 90th shift by the real count (real numbers don't vary).

In [17]:
p10 = FR.projection(inp, TEAM, n_games=10, sit_out=SIT)
pF = FR.projection(inp, TEAM, sit_out=SIT)
def team_line(p):
    t = p["team_range"]
    return {"window": f"{p['range']['start']} to {p['range']['end']}", "games": int(t["gp"]),
            "W": round(t["wins"], 2), "L": round(t["losses"], 2), "OTL": round(t["otl"], 2),
            "points": round(t["points"], 1), "points 10th-90th": f"{t['points_p10']:.0f} to {t['points_p90']:.0f}",
            "pace per 84": round(t["pace_84"], 1)}
display(pd.DataFrame([team_line(p10), team_line(pF)], index=["next 10 games", "rest of season"]))
t = pF["team_range"]
print(f"Full season: {t['final_points']:.1f} final points ({t['final_p10']:.0f} to {t['final_p90']:.0f}), "
      f"playoffs {t['playoff_pct']:.1f}%, division {t['division_pct']:.2f}%, Presidents' Trophy {t['presidents_pct']:.2f}%.")

,window,games,W,L,OTL,points,points 10th-90th,pace per 84
next 10 games,2026-11-16 to 2026-12-11,10,4.51,4.39,1.09,10.1,6 to 14,85.0
rest of season,2026-11-16 to 2027-04-10,63,26.67,29.34,6.99,60.3,47 to 74,80.4


Full season: 79.3 final points (66 to 93), playoffs 7.1%, division 0.13%, Presidents' Trophy 0.01%.


In [18]:
rng_cols = {"name": "Player", "sim_g": "G", "sim_a": "A", "sim_p": "P", "sim_p_p10": "P 10th", "sim_p_p90": "P 90th"}
ssn_cols = {"name": "Player", "real_p": "Real P", "g": "G", "a": "A", "p": "P", "p_p10": "P 10th", "p_p90": "P 90th"}
print("Next 10 games, range only:")
display(p10["team_skaters"]["range"][list(rng_cols)].rename(columns=rng_cols).round(1).head(6))
print("Rest of season, season to date (= projected final totals):")
display(pF["team_skaters"]["season"][list(ssn_cols)].rename(columns=ssn_cols).round(1).head(6))
print("Goalies, rest of season: range only, then season to date")
gr_cols = {"name": "Goalie", "role": "Role", "sim_gp": "GP", "sim_wins": "W", "sim_losses": "L", "sim_otl": "OTL", "sim_gaa": "GAA"}
display(pF["team_goalies"]["range"][list(gr_cols)].rename(columns=gr_cols).round(2))
display(pF["team_goalies"]["season"][list(g_season)].rename(columns=g_season).round(2))

Next 10 games, range only:


,Player,G,A,P,P 10th,P 90th
162,Lucas Raymond,3.7,6.2,9.9,5.0,15.0
163,Alex DeBrincat,4.0,4.4,8.4,4.0,13.0
174,Moritz Seider,1.1,4.9,6.1,3.0,10.0
166,Viktor Arvidsson,2.8,3.1,5.9,3.0,10.0
165,Andrew Copp,1.5,3.1,4.6,2.0,8.0
164,Emmitt Finnie,2.0,2.6,4.6,2.0,8.0


Rest of season, season to date (= projected final totals):


,Player,Real P,G,A,P,P 10th,P 90th
25,Lucas Raymond,21.0,30.5,51.1,81.6,66.0,98.0
57,Alex DeBrincat,16.0,31.3,35.7,67.0,54.0,81.0
117,Viktor Arvidsson,15.0,24.9,25.8,50.7,41.0,62.0
126,Moritz Seider,13.0,7.9,42.0,50.0,39.0,61.0
212,Emmitt Finnie,11.0,19.1,20.1,39.2,30.0,49.0
222,Andrew Copp,10.0,12.3,25.8,38.1,29.0,48.0


Goalies, rest of season: range only, then season to date


,Goalie,Role,GP,W,L,OTL,GAA
18,John Gibson,Starter,40,17.73,17.86,4.41,3.03
19,Daniil Tarasov,Backup,23,8.94,11.48,2.58,3.19


,Goalie,Role,GP,W,L,OTL,GA,GAA
14,John Gibson,Starter,53.0,21.73,24.86,5.41,162.13,3.15
46,Daniil Tarasov,Backup,32.0,11.94,14.48,4.58,88.34,2.84
79,Michal Postava,Not dressed,2.0,1.00,0.00,0.00,4.00,2.89


In [19]:
def projection_checks(p):
    real_s, _ = SE.real_lines(p["run"]["bundle_used"], p["range"]["start"])
    ss, rs, rr = p["season_skaters"].set_index("playerId"), p["range_skaters"].set_index("playerId"), real_s.set_index("playerId")
    ok = True
    for c in ["g", "a", "p"]:
        for q in ["", "_p10", "_p90"]:
            ok &= np.allclose(ss[f"{c}{q}"], rs[f"sim_{c}{q}"].reindex(ss.index).fillna(0) + rr[f"real_{c}"].reindex(ss.index).fillna(0),
                              rtol=0, atol=1e-9)
    return bool(ok)
for name, p in [("next 10", p10), ("rest of season", pF)]:
    print(f"{name}: season to date = real + range for every skater (average, 10th, 90th):", projection_checks(p))
assert projection_checks(p10) and projection_checks(pF)

next 10: season to date = real + range for every skater (average, 10th, 90th): True
rest of season: season to date = real + range for every skater (average, 10th, 90th): True


## 10. How much do lineup changes move things?

Each change is played against the baseline with **the same seed**, so every player has the same luck in both and the
comparison shows the change, not new random numbers. The game scores still differ (a stronger team changes every
game's odds), so each change comes with its noise: **± 2 standard errors** of the season-by-season difference. A
change smaller than its noise isn't really there.

For scale: Detroit's rest of season averages about 60 points from 63 games, and its 10th-90th range is about 27
points wide. One point is half a win.

In [20]:
def effect(label, p_base, p_new):
    d = FR.paired_change(p_base, p_new, TEAM)
    row = {"change": label, "points": round(d["points"], 2), "± 2 SE": round(d["points_2se"], 2)}
    if "playoff_pct" in d:
        row.update({"playoffs (% points)": round(d["playoff_pct"], 2), "playoffs ± 2 SE": round(d["playoff_pct_2se"], 2)})
    return row

rows = []
for rule in FR.MINUTES_RULES:
    rows.append(effect(f"(a) lines 1 and 4 swapped, rule '{rule}'", pF, FR.projection(inp, TEAM, sit_out=SIT, overrides=swap, minutes_rule=rule)))
callup = FR.replace(det, TEAM, pid("Carter Mazur"), pid("Carter Bear"), base)
rows.append(effect("(b) Carter Bear (AHL) dressed in Mazur's F12 slot", pF, FR.projection(inp, TEAM, sit_out=SIT, overrides=callup)))
rows.append(effect("extra: Tarasov starts, Gibson backs up (G1/G2 swap)", pF,
                   FR.projection(inp, TEAM, sit_out=SIT, overrides=FR.swap_slots(det, TEAM, "G1", "G2"))))
rows.append(effect("extra: Gibson's start share 0.80 (default 0.64)", pF, FR.projection(inp, TEAM, sit_out=SIT, goalie_share={TEAM: 0.8})))
rows.append(effect("(a) over the next 10 games only, rule 'moved'", p10, FR.projection(inp, TEAM, n_games=10, sit_out=SIT, overrides=swap)))
effects = pd.DataFrame(rows)
print("Detroit, rest of the season from", START, "(10,000 seasons each):")
effects

Detroit, rest of the season from 2026-11-16 (10,000 seasons each):


,change,points,± 2 SE,playoffs (% points),playoffs ± 2 SE
0,"(a) lines 1 and 4 swapped, rule 'moved'",-2.35,0.20,-2.50,0.59
1,"(a) lines 1 and 4 swapped, rule 'blend'",-0.66,0.20,-0.68,0.62
2,"(a) lines 1 and 4 swapped, rule 'slot'",-2.31,0.20,-2.31,0.58
3,(b) Carter Bear (AHL) dressed in Mazur's F12 slot,0.31,0.21,0.42,0.65
4,"extra: Tarasov starts, Gibson backs up (G1/G2 ...",-0.50,0.20,-0.42,0.63
5,extra: Gibson's start share 0.80 (default 0.64),0.33,0.20,0.84,0.66
6,"(a) over the next 10 games only, rule 'moved'",-0.28,0.07,NaN,NaN


In [21]:
bear = base[base["playerId"] == pid("Carter Bear")].iloc[0]
mazur = base[base["playerId"] == pid("Carter Mazur")].iloc[0]
print(f"(b) why it's small: Bear's projection {bear['proj_war_60']:.3f} WAR/60 vs. Mazur's {mazur['proj_war_60']:.3f}, "
      f"at about {mazur['minutes_per_game']:.1f} minutes a game: "
      f"{(bear['proj_war_60'] - mazur['proj_war_60']) * mazur['minutes_per_game'] / 60 * 63:.2f} WAR over 63 games.")
w10, wF = p10["team_range"], pF["team_range"]
print(f"(c) 10th-90th width of Detroit's points: next 10 games {w10['points_p10']:.0f} to {w10['points_p90']:.0f} "
      f"({w10['points_p90'] - w10['points_p10']:.0f} points wide, {w10['points']:.1f} on average); rest of season "
      f"{wF['points_p10']:.0f} to {wF['points_p90']:.0f} ({wF['points_p90'] - wF['points_p10']:.0f} wide, {wF['points']:.1f} on average).")
print(f"    Per game: {(w10['points_p90'] - w10['points_p10']) / w10['gp']:.2f} vs. {(wF['points_p90'] - wF['points_p10']) / wF['gp']:.2f} points of width per game played.")

(b) why it's small: Bear's projection 0.043 WAR/60 vs. Mazur's 0.019, at about 11.5 minutes a game: 0.30 WAR over 63 games.
(c) 10th-90th width of Detroit's points: next 10 games 6 to 14 (8 points wide, 10.1 on average); rest of season 47 to 74 (27 wide, 60.3 on average).
    Per game: 0.80 vs. 0.43 points of width per game played.


## 11. Speed

Timed on this Mac, with the update on and player lines included (the first run of each also builds the lineups).

In [22]:
timings = []
for label, f in [("story, next 10 games", lambda: FR.story(inp, TEAM, n_games=10, sit_out=SIT, seed=5)),
                 ("story, rest of season", lambda: FR.story(inp, TEAM, sit_out=SIT, seed=5)),
                 ("projection, next 10 games (10,000)", lambda: FR.projection(inp, TEAM, n_games=10, sit_out=SIT, seed=5)),
                 ("projection, rest of season (10,000)", lambda: FR.projection(inp, TEAM, sit_out=SIT, seed=5)),
                 ("projection, rest of season, lines swapped", lambda: FR.projection(inp, TEAM, sit_out=SIT, overrides=swap, seed=5))]:
    t0 = time.time()
    f()
    timings.append({"run": label, "seconds": round(time.time() - t0, 2)})
pd.DataFrame(timings)

,run,seconds
0,"story, next 10 games",0.66
1,"story, rest of season",0.63
2,"projection, next 10 games (10,000)",4.11
3,"projection, rest of season (10,000)",6.90
4,"projection, rest of season, lines swapped",6.78


## 12. Git: nothing new under data/

This notebook writes nothing. If the app later saves franchise lineups, the reserved path is
`data/franchise_lineups_2026-27.csv` (a top-level CSV in data/, so `.gitignore` already ignores it). A subfolder such
as `data/franchise/` would **not** be ignored by `data/*.csv`.

In [23]:
def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd="..").stdout.rstrip()
print("git check-ignore -v data/franchise_lineups_2026-27.csv ->", git("check-ignore", "-v", "data/franchise_lineups_2026-27.csv"))
print("git check-ignore -v data/franchise/lineups.csv ->", git("check-ignore", "-v", "data/franchise/lineups.csv") or "(not ignored)")
print("Tracked files under data/:", git("ls-files", "data").split())
print("git status --porcelain:")
print(git("status", "--porcelain"))

git check-ignore -v data/franchise_lineups_2026-27.csv -> .gitignore:2:data/*.csv	data/franchise_lineups_2026-27.csv
git check-ignore -v data/franchise/lineups.csv -> (not ignored)
Tracked files under data/: ['data/README.md']
git status --porcelain:
 M CLAUDE.md
 M app_pages/simulator.py
 M src/lineups.py
 M src/season.py
?? notebooks/30_franchise_mode.ipynb
?? src/franchise.py


## Takeaways

- **The engine reproduces the simulator exactly.**
  - With no overrides over the full season, franchise mode equals `season.run_scenario` bit for bit, with and
    without the update cache, for one season and for 10,000.
  - The edited `season.py` and `lineups.py` give the same results as the committed versions.
  - All 32 baseline slot tables, passed back as overrides, give the baseline minutes exactly.
- **The stories add up.** For all 32 teams: GF = the players' goals, points = 2W + OTL, and games played = the
  scheduled games in the window. Detroit's 10-game window has exactly 10 games. Season to date = real so far + the
  range for all 791 skaters and 80 goalies, and in the projections for the averages, 10th and 90th too.
- **For line moves, the minutes rule decides almost everything.** Swapping Detroit's 1st and 4th lines:
  - under your "moved" rule, it costs about **2.35 points (± 0.20)** and 2.5 points of playoff odds over the rest of
    the season, because Raymond goes from 19.4 to 12.5 minutes a game;
  - "slot only" is nearly the same (-2.31);
  - the 75/25 blend is about a third of that (-0.66), because players keep most of their own ice time.
- **Fourth-line changes are small because the minutes are small.** Carter Bear (AHL) in Mazur's F12 slot adds
  **0.31 points (± 0.21)**, about what the gap in their projections (0.043 vs. 0.019 WAR/60) at 11.5 minutes a game
  predicts: 0.3 WAR over 63 games.
- **Goalies:** making Tarasov the starter costs 0.50 points. Raising Gibson's start share to 0.80 adds 0.33 (± 0.20).
- **Short windows are mostly luck.** Over Detroit's next 10 games the projection is 10.1 points, with a 10th-90th
  range of 6 to 14 (8 points wide). Over the rest of the season it's 60.3, from 47 to 74 (27 wide).
  - Per game played, the 10-game range is about twice as wide (0.80 vs. 0.43 points).
  - The line swap costs only 0.28 points over 10 games, far inside that range.
  - One story is one draw: with the same lineup, seed 0 has Detroit going 3-4-3 and seed 1 going 1-8-1.
- **Speed:**
  - A one-season story takes about 0.5 s.
  - A 10,000-season projection takes about 3 s for 10 games and about 6.5 s for the rest of the season. About 2 s of
    that is player lines, and the rest is teams alone.
- **Ready for notebook 31:** every story keeps every pool player's random season: z, draw size, WAR/60 change,
  season WAR/60 and scoring multiplier (640 dressed today). z is a whole-season draw even in a 10-game window.
- **What it can't do:**
  - no chemistry or linemate effects;
  - no injuries; lineups stay fixed through the window;
  - no shots, save %, ice time or +/-.